# 03. Ablation Study

Study of the influence of components: BatchNorm, Dropout, Focal Loss.

In [ ]:
import torch
import sys
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from torchvision.ops import sigmoid_focal_loss

sys.path.append('..')
from rs.model import FocalLoss, PositionPredictor
from rs.training import train_model
from rs.evaluation import evaluate_fsr
from rs.channels import qsc_erasure_channel
from rs.dataset_gen import RSPositionDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parents[0]
table_out_dir = ROOT / "tables"
table_out_dir.mkdir(parents=True, exist_ok=True)

graph_out_dir = ROOT / "graphs"
graph_out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
P_ERR, P_ERASE = 0.02, 0.06
TRAIN_SIZE, EPOCHS = 50000, 150

print('Generating dataset...')
dataset = RSPositionDataset(TRAIN_SIZE, P_ERR, P_ERASE)
print(f'Generated {len(dataset)} examples.')

In [ ]:
class ConfigurableModel(nn.Module):
    def __init__(self, use_bn=True, dropout=0.1):
        super().__init__()
        layers = []
        in_size = 511
        for _ in range(4):
            layers.append(nn.Linear(in_size, 512))
            if use_bn: layers.append(nn.BatchNorm1d(512))
            layers.append(nn.ReLU())
            if dropout > 0: layers.append(nn.Dropout(dropout))
            in_size = 512
        layers.append(nn.Linear(512, 255))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x): return self.net(x)

def run_experiment(name, use_bn, dropout, loss_type, alpha=0.25, gamma=1.0):
    print(f'{name}...', end=' ')
    model = ConfigurableModel(use_bn=use_bn, dropout=dropout).to(device)
    
    if loss_type == 'BCE':
        criterion = nn.BCEWithLogitsLoss()
    else:
        criterion = lambda i, t: sigmoid_focal_loss(i, t, alpha=alpha, gamma=gamma, reduction='mean')
    
    train_model(model, dataset, criterion, epochs=EPOCHS, device=device, verbose=False)
    fsr = evaluate_fsr(model, qsc_erasure_channel, P_ERR, P_ERASE, device=device)
    print(f'FSR = {fsr:.1%}')
    return fsr, model

In [ ]:
sequential_results = []

# Baseline: BCE w/o regularization
fsr, _ = run_experiment('BCE (baseline)', use_bn=False, dropout=0.0, loss_type='BCE')
sequential_results.append({'config': 'BCE (baseline)', 'fsr': fsr})

# BCE + BN
fsr, _ = run_experiment('BCE + BN', use_bn=True, dropout=0.0, loss_type='BCE')
sequential_results.append({'config': 'BCE + BN', 'fsr': fsr})

# BCE + Dropout
fsr, _ = run_experiment('BCE + Dropout', use_bn=False, dropout=0.1, loss_type='BCE')
sequential_results.append({'config': 'BCE + Dropout', 'fsr': fsr})

# BCE + Dropout + BN
fsr, _ = run_experiment('BCE + Dropout + BN', use_bn=True, dropout=0.1, loss_type='BCE')
sequential_results.append({'config': 'BCE + Dropout + BN', 'fsr': fsr})

# Only Focal Loss
fsr, _ = run_experiment('Focal', use_bn=False, dropout=0.0, loss_type='Focal')
sequential_results.append({'config': 'Focal', 'fsr': fsr})

# Focal Loss + BN
fsr, _ = run_experiment('Focal + BN', use_bn=True, dropout=0.0, loss_type='Focal')
sequential_results.append({'config': 'Focal + BN', 'fsr': fsr})

# Focal Loss + Dropout
fsr, _ = run_experiment('Focal + Dropout', use_bn=False, dropout=0.1, loss_type='Focal')
sequential_results.append({'config': 'Focal + Dropout', 'fsr': fsr})

# Focal Loss + BN + Dropout
fsr, _ = run_experiment('Focal + Dropout + BN', use_bn=True, dropout=0.1, loss_type='Focal')
sequential_results.append({'config': 'Focal + Dropout + BN', 'fsr': fsr})

In [ ]:
df_seq = pd.DataFrame(sequential_results)
df_seq.to_csv(table_out_dir / 'ablation_results.csv', index=False)

print('Последовательное добавление компонентов:')
print(df_seq.to_string(index=False))

In [ ]:
focal_params = [
    (0.15, 1.5), (0.15, 2.0), 
    (0.20, 1.5), (0.20, 2.0), 
    (0.25, 1.0), (0.30, 1.0)
]

focal_results = []
for alpha, gamma in focal_params:
    name = f'α={alpha}, γ={gamma}'
    fsr, _ = run_experiment(name, use_bn=True, dropout=0.1, loss_type='Focal', alpha=alpha, gamma=gamma)
    focal_results.append({'alpha': alpha, 'gamma': gamma, 'fsr': fsr})

In [ ]:
df_focal = pd.DataFrame(focal_results)
df_focal.to_csv(table_out_dir / 'focal_tuning.csv', index=False)

print('Focal Loss tuning:')
print(df_focal.to_string(index=False))

best_focal = df_focal.loc[df_focal['fsr'].idxmax()]
print(f"\nBest parameters: α={best_focal['alpha']}, γ={best_focal['gamma']}, FSR={best_focal['fsr']:.1%}")

In [ ]:
print('Final model training...')
final_model = ConfigurableModel(use_bn=True, dropout=0.1).to(device)
criterion = FocalLoss(0.2, 1.5)
train_model(final_model, dataset, criterion, epochs=EPOCHS, device=device, verbose=False)
print('Training finished.')

In [ ]:
print('Threshold tuning...')

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
results = []

for theta in thresholds:
    fsr = evaluate_fsr(final_model, qsc_erasure_channel, P_ERR, P_ERASE, threshold=theta, device=device)

    print(f'threshold = {theta}: {fsr}')
    results.append({
        "threshold": theta,
        "fsr": fsr
    })

In [ ]:
df = pd.DataFrame(results)
df.to_csv(table_out_dir / "threshold.csv", index=False)

print(df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

best_idx = df["fsr"].idxmax()
best_threshold = df.loc[best_idx, "threshold"]
best_fsr = df.loc[best_idx, "fsr"]

plt.figure(figsize=(8, 5))

plt.plot(df["threshold"], df["fsr"] * 100, marker="o", linewidth=2)

plt.axvline(best_threshold, linestyle="--", linewidth=2, label="Optimum")

plt.xlabel(r"Threshold $\theta$")
plt.ylabel("FSR, %")
plt.title("Threshold tuning for hybrid decoder")

plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig(graph_out_dir / 'threshold.png', dpi=150)
plt.show()